In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("To_clean.csv") 

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
df.rename(columns={'Application St Addresss': 'Application St Address'}, inplace=True)

In [ ]:
#  Check missing values for each column 
missing_summary = df.isnull().sum().to_frame(name='Missing_Values')
missing_summary['Missing_%'] = (missing_summary['Missing_Values'] / len(df) * 100).round(2)

# Check number of unique values for each column 
unique_summary = df.nunique().to_frame(name='Unique_Values')

# Combine summaries 
summary = missing_summary.join(unique_summary)

#Display summary
print(summary)

In [ ]:
df.info()

In [ ]:
# Number of duplicate rows
duplicate_count = df.duplicated().sum()
print("\nNumber of duplicate rows:", duplicate_count)

In [ ]:
df = df.drop_duplicates()
duplicate_count = df.duplicated().sum()
print("\nNumber of duplicate rows:", duplicate_count)

In [ ]:
print(df['ID Number'].value_counts()[df['ID Number'].value_counts() > 1])


In [ ]:
df.columns

In [ ]:
del df['Source']

## Application Addresses

Application Addresses were cleaned in google sheet, some of the errors are resolved below

In [ ]:
df[df['Application St Address'] == '1 Millbrook Lane Unit 213']
# Apply conditional update
df.loc[df['Application St Address'] == '1 Millbrook Lane Unit 213', 'Application St Address'] = '1 Millbrook Lane'
df.loc[df['Application St Address'] == '1 Millbrook Lane', 'Application Unit'] = 'Unit 213'

df[df['Application St Address'] == '1 Millbrook Lane Unit 213 ']
df.loc[df['Application St Address'] == '1 Millbrook Lane Unit 213 ', 'Application St Address'] = '1 Millbrook Lane'
df.loc[df['Application St Address'] == '1 Millbrook Lane', 'Application Unit'] = 'Unit 213'


## Current Residence Column 

In [ ]:
print(f"Unique current_residence formats (sample): {len(df['Current Residence'].unique())}")
print("Rows with comma:", df['Current Residence'].str.contains(",", na=False).sum())

In [ ]:
# function to extract city and state from the 'Current Residence' column
import re
def extract_city_state(entry):
    if pd.isna(entry):
        return pd.Series([None, None])
    
    entry = entry.strip()

    # Try to split by comma
    if ',' in entry:
        parts = entry.split(',')
        city = parts[0].strip()
        state = parts[1].strip() if len(parts) > 1 else 'MA'
        return pd.Series([city, state])

    # Try to match City State pattern (e.g., "Springfield MA")
    match = re.match(r'([A-Za-z\s]+)\s+([A-Z]{2})$', entry)
    if match:
        city = match.group(1).strip()
        state = match.group(2).strip()
        return pd.Series([city, state])
    
    # If only city is present, assume MA
    return pd.Series([entry.title(), 'MA'])


In [ ]:
# applying the function to the 'Current Residence' column

df[['Current Residence City', 'Current Residence State']] = df['Current Residence'].apply(extract_city_state)

# Preview results
print(df[['Current Residence', 'Current Residence City', 'Current Residence State']].head())
# Check for any remaining rows with missing city or state
missing_city_state = df[df['Current Residence City'].isnull() | df['Current Residence State'].isnull()]
print(f"Rows with missing city or state: {len(missing_city_state)}")

#print the row with the missin city or state
print(missing_city_state)

# Save cleaned data to a new CSV file
#df.to_csv("To_clean_cleaned.csv", index=False)

## Submission Date/Time 

In [ ]:
#change submission date/time to separate columns 
# Convert the string to a pandas datetime object
df['Submission Date'] = pd.to_datetime(df['Submission Date'])
#  Extract the date and time components
df['submission_date'] = df['Submission Date'].dt.date
df['submission_time'] = df['Submission Date'].dt.time

print(df[['Submission Date', 'submission_date', 'submission_time']].head())

## Race and Ethnicity

In [ ]:
# Standardizing the race/ethnicity column
df['Race/Ethnicity'].describe()

In [ ]:
df['Race/Ethnicity'].value_counts().head(10)

In [ ]:
# Define lowercased canonical race labels based on CHAPA submission form 
canonical_races = [
    "Asian",
    "Black or African American",
    "Hispanic or Latino",
    "Native American or Alaskan Native",
    "Middle Eastern or North African",
    "Pacific Islander or Native Hawaiian",
    "White",
    "Choose not to answer",
    "Other"
]

# Custom aliases (lowercased): free-text → canonical label
race_aliases = {
    "hispaniclatino": "Hispanic or Latino",
    "hispanic latino": "Hispanic or Latino",
    "hispanic/latino": "Hispanic or Latino",
    "latino": "Hispanic or Latino",
    "latin american": "Hispanic or Latino",
    "pacific islander": "Pacific Islander or Native Hawaiian",
    "native hawaiian": "Pacific Islander or Native Hawaiian",
    "black": "Black or African American",
    "african american": "Black or African American",
    "navajo": "Native American or Alaskan Native",
    "native american": "Native American or Alaskan Native",
    "cherokee": "Native American or Alaskan Native",
    "north african": "Middle Eastern or North African",
    "middle eastern": "Middle Eastern or North African",
    "egyptian": "Middle Eastern or North African",
    "afghan": "Asian",
    "brazilian": "Hispanic or Latino",
    "italian": "White",
    "greek": "White"
    # add more as needed...
}

canonical_races_lower = [r.lower() for r in canonical_races]

# Function to extract matched races and unmatched free text
def extract_race_with_logging(text):
    if pd.isna(text):
        return pd.Series([[], None])

    text = text.lower()
    matched = []
    text_copy = text  # we’ll remove matched terms from this progressively

    for label, label_lc in zip(canonical_races, canonical_races_lower):
        # Regex: match exact label with word boundaries
        pattern = r'\b' + re.escape(label_lc) + r'\b'
        if re.search(pattern, text_copy):
            matched.append(label)
            text_copy = re.sub(pattern, '', text_copy)  # remove the matched part

    # 2. Match aliases using substring (not strict regex for flexibility)
    for alias, canonical in race_aliases.items():
        if alias in text_copy and canonical not in matched:
            matched.append(canonical)
            text_copy = text_copy.replace(alias, '')
            
    # After removing all known labels, clean up the rest to find unmatched text
    leftover = re.sub(r'[^a-zA-Z ]+', '', text_copy)  # remove punctuation/numbers
    leftover = re.sub(r'\s+', ' ', leftover).strip()  # normalize spaces

    return pd.Series([matched, leftover if leftover else None])



In [ ]:
# Apply the function to dataset 
df[['race_list_chapa', 'unmatched_text']] = df['Race/Ethnicity'].apply(extract_race_with_logging)

#categorize the responses just for initial analysis (deleted later)
def classify_race(x):
    if len(x) == 0:
        return 'Missing'
    elif len(x) == 1:
        return 'Single'
    else:
        return 'Multiple'
    
df['race_response_type'] = df['race_list_chapa'].apply(classify_race)



In [ ]:
# View entries with unmatched text
unmatched_df = df[df['unmatched_text'].notna()]
print(unmatched_df[['Race/Ethnicity', 'race_list_chapa', 'unmatched_text']].head())

# Save if needed
unmatched_df.to_csv("unmatched_race_entries.csv", index=False)

### Multiple Race Selection

In [ ]:
# Calculate the number of races in each entry
df['num_races_selected'] = df['race_list_chapa'].apply(len)

# Get the maximum
max_races_selected = df['num_races_selected'].max()
print(max_races_selected)

In [ ]:
# Create a variable with race list as a string for easier viewing
df['race_list_string'] = df['race_list_chapa'].apply(lambda x: ', '.join(x) if isinstance(x, list) else '')


In [ ]:
# Normalize order by splitting and sorting so its easy to categorize combinations
def normalize_combination(race_string):
    if pd.isna(race_string) or race_string.strip() == '':
        return 'Missing'
    parts = [part.strip() for part in race_string.split(',')]
    parts_sorted = sorted(parts)
    return ', '.join(parts_sorted)

# Apply normalization to a new column
df['race_combination_normalized'] = df['race_list_string'].apply(normalize_combination)

# Count unique combinations (ignoring order)
combination_counts = df['race_combination_normalized'].value_counts()

# View result
print(combination_counts)
len(df['race_combination_normalized'].unique())

In [ ]:
# If someone answered "Choose not to answer, Hispanic or Latino", we want to normalize it to just "Hispanic or Latino"
df.loc[df['race_combination_normalized'] == 'Choose not to answer, Hispanic or Latino', 'race_combination_normalized'] = 'Hispanic or Latino'
df['race_combination_normalized'].value_counts()

### Manually Matching the Unmatched entries for Race

In [ ]:
df.loc[df['race_combination_normalized'] == 'Missing', 
       ['Race/Ethnicity', 'race_list_chapa', 'unmatched_text', 'race_response_type', 'race_list_string', 'race_combination_normalized']]


In [ ]:
# Manually changing race normalized values 
df.loc[df['unmatched_text'] == 'iraqiamerican', 'race_combination_normalized'] = 'Middle Eastern or North African'
df.loc[df['unmatched_text'] == 'south asia pakistan', 'race_combination_normalized'] = 'Asian'
df.loc[df['unmatched_text'] == 'europeanafrican', 'race_combination_normalized'] = 'Black or African American, White'
df.loc[df['unmatched_text'] == 'cape verde portuguese', 'race_combination_normalized'] = 'Black or African American'
df.loc[df['unmatched_text'] == 'uyghur', 'race_combination_normalized'] = 'Asian'
df.loc[df['unmatched_text'] == 'cape verdean', 'race_combination_normalized'] = 'Black or African American'
df.loc[df['unmatched_text'] == 'pakistani ukrainian', 'race_combination_normalized'] = 'Asian, White'

df.loc[df['race_combination_normalized'] == 'Missing', 
       ['Race/Ethnicity', 'race_list_chapa', 'unmatched_text', 'race_response_type', 'race_list_string', 'race_combination_normalized']]

In [ ]:
df['race_combination_normalized'].value_counts() # 26 unique combination 

### Creating a Simplified Race column for ease of analysis

In [ ]:
def simplify_race(race_str):
    if pd.isna(race_str) or race_str.strip() in ['Missing', 'Choose not to answer', 'Other']:
        return 'Choose not to answer/Missing/Other'

    # Count how many race groups are included
    races = [r.strip() for r in race_str.split(',')]
    
    if len(races) > 1:
        return 'Multiple Races/Ethnicities'
    
    # If only one race is present, return the simplified category
    if 'White' in race_str:
        return 'White'
    elif 'Black or African American' in race_str:
        return 'Black or African American'
    elif 'Hispanic or Latino' in race_str:
        return 'Hispanic or Latino'
    elif 'Middle Eastern or North African' in race_str:
        return 'Middle Eastern or North African'
    elif 'Asian' in race_str:
        return 'Asian'
    elif 'Native American or Alaskan Native' in race_str:
        return 'Native American or Alaskan Native'
    else:
        return 'Choose not to answer/Missing/Other'


In [ ]:
# Apply function to create new column
df['Race_Simplified'] = df['race_combination_normalized'].apply(simplify_race)

In [ ]:
df['Race_Simplified'].value_counts()

### Race/Ethnicity: Census Data 

In [ ]:
# Census Race Categories

# Custom aliases (lowercased): free-text → canonical label
census_race_aliases = {
    'Choose not to answer': 'Missing',
    'White': 'White alone',
    'Black or African American': 'Black or African American alone',
    'Asian, Pacific Islander or Native Hawaiian': 'Asian; Native Hawaiian or Other Pacific Islander',
    'Hispanic or Latino': 'Missing',
    'Missing': 'Missing',
    'Black or African American, Hispanic or Latino': 'Black or African American',
    'Asian, Black or African American, Pacific Islander or Native Hawaiian, White':'Asian, Black or African American, Native Hawaiian or Other Pacific Islander, White',
    'Middle Eastern or North African': 'White alone',
    'Asian, Pacific Islander or Native Hawaiian, White': 'Asian; Native Hawaiian or Other Pacific Islander; White',
    'Hispanic or Latino, White': 'White alone',
    'Black or African American, White': 'Black or African American; White',
    'Black or African American, Native American or Alaskan Native, White': 'Black or African American; American Indian and Alaska Native; White',
    'Asian': 'Asian alone',
    'Black or African American, Native American or Alaskan Native': 'Black or African American; American Indian and Alaska Native',
    'Asian, Black or African American, Middle Eastern or North African, White': 'Asian; Black or African American; White',
    'Middle Eastern or North African, White': 'White alone',
    'Asian, White': 'Asian; White',
    'Native American or Alaskan Native': 'American Indian and Alaska Native alone',
    'Other': 'Some Other Race',
    'Hispanic or Latino, Native American or Alaskan Native': 'Hispanic or Latino; American Indian and Alaska Native',
    'Native American or Alaskan Native, White': 'American Indian and Alaska Native; White',
    'Asian, Middle Eastern or North African, White': 'Asian; White',
    'Asian, Middle Eastern or North African': 'Asian; White',
    'Black or African American, Hispanic or Latino, White': 'Black or African American; White',
    'Pacific Islander or Native Hawaiian': 'Native Hawaiian or Other Pacific Islander alone',
}

# Define the function
def map_to_census_category(race_combo):
    # Use dictionary lookup, return 'Unmapped' if no match found
    return census_race_aliases.get(race_combo, 'Unmapped')

# Apply it to your DataFrame
df['census_races'] = df['race_combination_normalized'].apply(map_to_census_category)


## Household Type

In [ ]:
# Houshold Type cleaning 
df['HH Type'].unique()

### Creating a Function to normalize household types 

In [ ]:
canonical_hh_types = [
    "Single person",
    "Married/partners/couple, no dependents",
    "Married/partners/couple, with dependents",
    "Single parent",
    "Other", 
    "More than one related adults, with dependents",
    "More than one related adults, no dependents"
]

hh_aliases = {
    "single person": "Single person",
    "living alone": "Single person",
    "solo": "Single person",
    "separated": "Single person",
    "siblings": "More than one related adults, no dependents",
    
    "more than one related adults, with dependents": "More than one related adults, with dependents",
    "more than one related adults, no dependents": "More than one related adults, with dependents",
    
    "married": "Married/partners/couple, no dependents",  # base case
    "married no kids": "Married/partners/couple, no dependents",
    "couple no children": "Married/partners/couple, no dependents",
    "partnered no dependents": "Married/partners/couple, no dependents",

    "married with kids": "Married/partners/couple, with dependents",
    "couple with dependents": "Married/partners/couple, with dependents",
    "married children": "Married/partners/couple, with dependents",
    "partnered with kids": "Married/partners/couple, with dependents",

    "single mom": "Single parent",
    "single dad": "Single parent",
    "single parent": "Single parent",
    "solo parent": "Single parent",

    "other": "Other"
    # add more free-text variants as needed
}

def extract_hh_type(text):
    if pd.isna(text):
        return pd.Series([None, None])
    
    text = text.lower().strip()

    matched = None
    for alias, canonical in hh_aliases.items():
        if alias in text:
            matched = canonical
            break

    leftover = None if matched else text  # keep unmatched as leftover
    return pd.Series([matched, leftover])

# Apply the function to the 'HH Type' column
df[['hh_type_standardized', 'hh_unmatched']] = df['HH Type'].apply(extract_hh_type)

# Preview unmatched free-text entries
df[df['hh_unmatched'].notna()][['HH Type', 'hh_type_standardized','hh_unmatched']]

### Manually changing household types that don't match

In [ ]:
# Manually changing HH Type  values 
df.loc[df['hh_unmatched'] == 'two person hh', 'hh_type_standardized'] = 'More than one related adults, no dependents'
df.loc[df['hh_unmatched'] == 'currently in divorce process', 'hh_type_standardized'] = 'Single person'
df.loc[df['hh_unmatched'] == 'domestic partners 1 child 13yrs', 'hh_type_standardized'] = 'Married/partners/couple, with dependents'
df.loc[df['hh_unmatched'] == 'live with parents', 'hh_type_standardized'] = 'More than one related adults, no dependents'
df.loc[df['hh_unmatched'] == 'single, would like to foster/adopt', 'hh_type_standardized'] = 'Single person'
df.loc[df['hh_unmatched'] == 'parents/sisters', 'hh_type_standardized'] = 'More than one related adults, no dependents'
df.loc[df['hh_unmatched'] == 'partners', 'hh_type_standardized'] = 'Married/partners/couple, no dependents'
df.loc[df['hh_unmatched'] == 'engaged with a child 16 months old', 'hh_type_standardized'] = 'Married/partners/couple, with dependents'
df.loc[df['hh_unmatched'] == 'living with parents', 'hh_type_standardized'] = 'More than one related adults, no dependents'
df.loc[df['hh_unmatched'] == 'single/living with parents, with dependents', 'hh_type_standardized'] = 'More than one related adults, with dependents'
df.loc[df['hh_unmatched'] == 'daughter and father', 'hh_type_standardized'] = 'Single parent'
df.loc[df['hh_unmatched'] == 'single, with live in aide', 'hh_type_standardized'] = 'Single person'
df.loc[df['hh_unmatched'] == 'divorced', 'hh_type_standardized'] = 'Single person'
df.loc[df['hh_unmatched'] == 'my aunt &uncle they live with me', 'hh_type_standardized'] = 'More than one related adults, no dependents'
df.loc[df['hh_unmatched'] == 'my aunt & uncle they live with me.', 'hh_type_standardized'] = 'More than one related adults, no dependents'


# Preview unmatched free-text entries
df[df['hh_type_standardized'].isna()][['HH Type', 'hh_type_standardized','hh_unmatched']]

## Disability

In [ ]:
df['Disability'] = df['Disability'].str.strip().str.title()

## Household Income and Assets 

In [ ]:
# changing the HH income and assets columns to numeric (Creating new columns)
df['HH income'] = pd.to_numeric(df['HH Income'].str.replace(',', ''), errors='coerce')
df['HH assets'] = pd.to_numeric(df['HH Assets'].str.replace(',', ''), errors='coerce')


In [ ]:
# Create new column: 1 if HH income is missing, 0 if not
df['hh_income_missing'] = df['HH income'].isna().astype(int)

# Create new column: 1 if HH assets is missing, 0 if not
df['hh_assets_missing'] = df['HH assets'].isna().astype(int)

# View counts of missing vs not missing
print("Missing HH Income:")
print(df['hh_income_missing'].value_counts())

print("\nMissing HH Assets:")
print(df['hh_assets_missing'].value_counts())


In [ ]:
# Calculate medians
hh_income_median = df['HH income'].median()
hh_assets_median = df['HH assets'].median()

# Fill missing values with median
df['HH income'] = df['HH income'].fillna(hh_income_median)
df['HH assets'] = df['HH assets'].fillna(hh_assets_median)

# Check value counts or head
print("Filled HH Income (first few rows):")
print(df['HH income'].head())

print("\nFilled HH Assets (first few rows):")
print(df['HH assets'].head())


### Creating Income Brackets

In [ ]:
# Define the max income to create the bins
max_income = df['HH income'].max()
print(f"Maximum HH income: ${max_income}")
print(f"Last Income bucket max value: ${int(max_income + 20000)}")

# Create bins: from 0 to the next 20k above max_income, in steps of 20k
bins = list(range(0, int(max_income + 20000), 20000))

# Create labels for each bin
labels = [f"${bins[i]}-${bins[i+1]-1}" for i in range(len(bins)-1)]

# Create the bracket column
df['income_bracket'] = pd.cut(df['HH income'], bins=bins, labels=labels, right=False)
df['income_bracket'].value_counts().sort_index()

### Creating Asset Brackets 

In [ ]:
# Clean and convert to numeric
df['HH assets'] = df['HH assets'].astype(str).str.replace(',', '')
df['HH assets'] = pd.to_numeric(df['HH assets'], errors='coerce')

#  Define asset bins with $20,000 steps and final bin as $300,000+
upper_cap = 300000
step = 20000

asset_bins = list(range(0, upper_cap, step)) + [float('inf')]

# Create labels like "$0–$19,999", ...
asset_labels = [f"${asset_bins[i]:,}-${asset_bins[i+1]-1:,.0f}" for i in range(len(asset_bins)-2)]
asset_labels.append("$300,000+")

#  Assign brackets to new column
df['asset_bracket'] = pd.cut(
    df['HH assets'],
    bins=asset_bins,
    labels=asset_labels,
    right=False
)

# View distribution
print(df['asset_bracket'].value_counts().sort_index())



## Save to CSV

In [ ]:
# save to a new CSV file
df.to_csv("To_clean_cleaned.csv", index=False)

In [ ]:
#drop all old columns and save to new CSV file 
df_cleaned = df.drop(columns=['HH Income', 'HH Assets', 'Submission Date', 'Application Property', 'Current Residence', "HH Type", "unmatched_text",'race_response_type','num_races_selected', 'race_list_string','hh_unmatched', 'hh_income_missing', 'hh_assets_missing' , 'race_list_chapa'])
#Rename columns for clarity
df_cleaned.rename(columns={
    'Current Residence City': 'Current Residence Town/City',
    'Current Residence State': 'Current Residence State',
    'submission_date': 'Submission Date',
    'submission_time': 'Submission Time',
    'race_combination_normalized': 'Race Combination Normalized',
    'census_races': 'Census Races',
    'hh_type_standardized': 'HH Type Standardized',
    'HH income': 'HH Income',
    'HH assets': 'HH Assets',
    'income_bracket':'Income Bracket',
    'Race_Simplified': 'Race Simplified',
    'asset_bracket': 'Asset Bracket'}, inplace=True)

df_cleaned.to_csv("Cleaned_Saniya_23_25.csv", index=False)


In [ ]:
df_cleaned.shape


## Duplicate ID rows; Creating a separate dataset for demographic analysis 

### Check for duplicates ignoring Submission Date and Time 
Some applicants have submitted same information at different times, we have kept the first entry for such duplicates

In [ ]:
# Define columns to ignore
ignore_cols = ['Submission Date', 'Submission Time']

# Define columns to use for checking duplicates
check_cols = [col for col in df_cleaned.columns if col not in ignore_cols]

# Count duplicate rows excluding the ignored columns
duplicate_count = df_cleaned.duplicated(subset=check_cols).sum()
print(f"Number of duplicate rows (ignoring {ignore_cols}):", duplicate_count)


In [ ]:
# View all rows that are duplicates (ignoring certain columns)
duplicate_rows = df_cleaned[df_cleaned.duplicated(subset=check_cols, keep=False)]
duplicate_rows.head()



In [ ]:
# Drop duplicate rows (ignoring 'Submission Date' and 'Submission Time' and keep first entry) only for demographics analysis dataset
df_cleaned_dem = df_cleaned.drop_duplicates(subset=check_cols, keep='first')


### Duplicate Ids

In [ ]:
# Find duplicate IDs (keep=False shows *all* duplicates, not just the later ones)
duplicate_id_rows = df_cleaned_dem[df_cleaned_dem['ID Number'].duplicated(keep=False)]

# sort by ID for clarity
duplicate_id_rows = duplicate_id_rows.sort_values(by='ID Number')



In [ ]:
print(df_cleaned_dem['ID Number'].value_counts()[df_cleaned_dem['ID Number'].value_counts() > 1])


In [ ]:
num_ids_with_more_than_two = (df_cleaned_dem['ID Number'].value_counts() > 2).sum()
print("Number of unique IDs with more than 2 entries:", num_ids_with_more_than_two)


In [ ]:
duplicate_id_rows.to_csv("duplicate_id_rows.csv", index=False)

### Dealing with all Duplicate IDs by keeping the first entry

In [ ]:

# Keep only the first occurrence of each duplicate ID Number
df_deduplicated = df_cleaned_dem.drop_duplicates(subset='ID Number', keep='first')
df_deduplicated.shape

In [ ]:
# Export to CSV
df_deduplicated.to_csv('./Summary Stats/demographic_data.csv', index=False)
